<div align="center">
<p align="center" style="width: 100%;">
    <img src="https://raw.githubusercontent.com/vlm-run/.github/refs/heads/main/profile/assets/vlm-black.svg" alt="VLM Run Logo" width="80" style="margin-bottom: -5px; color: #2e3138; vertical-align: middle; padding-right: 5px;"><br>
</p>
<p align="center"><a href="https://docs.vlm.run"><b>Website</b></a> | <a href="https://docs.vlm.run/"><b>API Docs</b></a> | <a href="https://docs.vlm.run/blog"><b>Blog</b></a> | <a href="https://discord.gg/AMApC2UzVY"><b>Discord</b></a> | <a href="https://chat.vlm.run"><b>Chat</b></a>
</p>
</div>

# VLM Run Orion - 3D Reconstruction (Node.js)

This comprehensive cookbook demonstrates [VLM Run Orion's](https://vlm.run/orion) 3D reconstruction capabilities using **Node.js/TypeScript**. For more details on the API, see the [Agent API docs](https://docs.vlm.run/agents/introduction).

For this notebook, we'll cover how to use the **VLM Run Agent Chat Completions API** - an OpenAI-compatible interface for building powerful 3D reconstruction workflows with the same familiar chat-completions interface.

We'll cover the following topics:
 1. 3D Reconstruction from Single Images (depth estimation and geometry inference)
 2. Multi-Object 3D Reconstruction from a Single Image
 3. 3D Reconstruction from Multiple Images/Video (multi-view stereo reconstruction)

## Prerequisites

- Node.js 18+
- VLM Run API key (get one at [app.vlm.run](https://app.vlm.run))
- Deno or tslab kernel for running TypeScript in Jupyter


## Setup

First, install the required packages and configure the environment.


In [1]:
// Install the VLM Run SDK
// npm install vlmrun openai zod zod-to-json-schema

// If using Deno kernel, install dependencies via npm specifiers
// For tslab, run: npm install vlmrun openai zod zod-to-json-schema in your project directory


In [2]:
// Import the VLM Run SDK and dependencies
import { VlmRun } from "npm:vlmrun";
import { z } from "npm:zod";
import { zodToJsonSchema } from "npm:zod-to-json-schema";


In [3]:
// Get API key from environment variable
const VLMRUN_API_KEY = Deno.env.get("VLMRUN_API_KEY");

if (!VLMRUN_API_KEY) {
    throw new Error("Please set the VLMRUN_API_KEY environment variable");
}

console.log("✓ API Key loaded successfully");


✓ API Key loaded successfully


## Initialize the VLM Run Client

We use the OpenAI-compatible chat completions interface through the VLM Run SDK.


In [ ]:
// Initialize the VLM Run client using the SDK
const client = new VlmRun({
    apiKey: VLMRUN_API_KEY,
    baseURL: "https://agent.vlm.run/v1"  // Use the agent API endpoint
});

console.log("✓ VLM Run client initialized successfully!");
console.log("Base URL: https://agent.vlm.run/v1");
console.log("Model: vlmrun-orion-1");


✓ VLM Run client initialized successfully!
Base URL: https://agent.vlm.run/v1
Model: vlmrun-orion-1


## Response Models (Schemas)

We define Zod schemas for structured outputs. These schemas provide type-safe, validated responses.


In [5]:
// 3D Reconstruction Response Schema
const Recon3DResponseSchema = z.object({
    recon_path: z.string().describe("Pre-signed URL to the 3D reconstruction file (PLY format)")
});

console.log("✓ Response schemas defined successfully!");
console.log("Schemas include type-safe validation for structured outputs.");


✓ Response schemas defined successfully!
Schemas include type-safe validation for structured outputs.


## Helper Functions

We create helper functions to simplify making chat completion requests and working with 3D reconstruction files.


In [6]:
/**
 * Make a chat completion request with optional images/video and structured output.
 * 
 * @param prompt - The text prompt/instruction
 * @param images - Optional list of images to process (URLs)
 * @param video - Optional video URL to process
 * @param responseSchema - Optional Zod schema for structured output
 * @param model - Model to use (default: vlmrun-orion-1:auto)
 * @returns Object with parsed data and full response (for extracting sessionId)
 */
async function chatCompletion(
    prompt,
    images,
    video,
    responseSchema,
    model = "vlmrun-orion-1:auto"
) {
    const content = [];
    content.push({ type: "text", text: prompt });

    if (images) {
        for (const image of images) {
            if (typeof image === "string") {
                if (!image.startsWith("http")) {
                    throw new Error("Image URLs must start with http or https");
                }
                content.push({
                    type: "image_url",
                    image_url: { url: image, detail: "auto" }
                });
            }
        }
    }

    if (video) {
        if (!video.startsWith("http")) {
            throw new Error("Video URLs must start with http or https");
        }
        content.push({
            type: "video_url",
            video_url: { url: video }
        });
    }

    const kwargs = {
        model: model,
        messages: [{ role: "user", content: content }]
    };

    if (responseSchema) {
        kwargs.response_format = {
            type: "json_schema",
            schema: zodToJsonSchema(responseSchema)
        };
    }

    const response = await client.agent.completions.create(kwargs);
    const responseText = response.choices[0].message.content || "";

    let parsed = null;
    if (responseSchema) {
        parsed = JSON.parse(responseText);
        parsed = responseSchema.parse(parsed);
    }

    return {
        data: parsed || responseText,
        response: response
    };
}

/**
 * Get a PLY file artifact using session ID and object ID.
 * 
 * @param sessionId - Session ID from the chat completion response
 * @param objectId - Object ID extracted from the message content (format: recon_XXXXXX)
 * @returns The artifact file data as Buffer
 */
async function getPlyArtifact(sessionId, objectId) {
    console.log(`Retrieving PLY artifact → ${objectId}`);
    console.log(`Session ID: ${sessionId}`);
    
    const artifact = await client.artifacts.get({
        sessionId: sessionId,
        objectId: objectId,
        rawResponse: true
    });
    
    if (!(artifact instanceof Uint8Array || artifact instanceof Buffer)) {
        throw new Error("Expected artifact to be a Buffer or Uint8Array");
    }
    
    const data = artifact instanceof Buffer ? artifact : Buffer.from(artifact);
    console.log(`✓ Retrieved ${data.length} bytes`);
    
    return data;
}

console.log("✓ Helper functions defined!");
console.log("  - chatCompletion(): Make chat completion requests with images/video");
console.log("  - getPlyArtifact(): Get 3D reconstruction PLY files using artifacts API");


✓ Helper functions defined!
  - chatCompletion(): Make chat completion requests with images/video
  - getPlyArtifact(): Get 3D reconstruction PLY files using artifacts API


### 1. 3D Reconstruction from a Single Image

Create a 3D model from a single image. The model will infer depth and geometry to create a full 3D reconstruction.


In [7]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/guided-segmentation/image-11.png";

console.log("Input image:", IMAGE_URL);
console.log("\n>> Generating 3D reconstruction...");

const result = await chatCompletion(
    "Generate a 3D reconstruction of the table in the image",
    [IMAGE_URL],
    undefined,
    Recon3DResponseSchema
);

console.log("\n>> RESPONSE");
console.log(result.data);
console.log("\n>> 3D Reconstruction PLY file:", result.data.recon_path);

// Extract session ID and object ID
const sessionId = result.response.session_id;
const messageContent = result.response.choices[0]?.message?.content || "";

// Try to extract objectId from message content (format: recon_XXXXXX)
let reconMatch = messageContent.match(/recon_[a-f0-9]{6}/);

// If not found in message content, try to extract from the URL path
if (!reconMatch && result.data.recon_path) {
    // Extract UUID from URL path and convert to objectId format
    // URL format: .../artifacts/{sessionId}/{uuid}/filename.ply
    const urlMatch = result.data.recon_path.match(/\/artifacts\/[^\/]+\/([^\/]+)\//);
    if (urlMatch) {
        // Convert UUID to objectId format (take first 6 hex chars)
        const uuid = urlMatch[1].replace(/-/g, '');
        const hexId = uuid.substring(0, 6);
        reconMatch = [`recon_${hexId}`];
    }
}

if (!sessionId) {
    throw new Error("Session ID not found in response");
}

if (!reconMatch) {
    throw new Error("Object ID (recon_XXXXXX) not found in message content or URL");
}

const objectId = reconMatch[0];
console.log("\n>> Extracted IDs:");
console.log(`  Session ID: ${sessionId}`);
console.log(`  Object ID: ${objectId}`);


Input image: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/guided-segmentation/image-11.png

>> Generating 3D reconstruction...

>> RESPONSE
{ recon_path: "recon_d98a8d" }

>> 3D Reconstruction PLY file: recon_d98a8d

>> Extracted IDs:
  Session ID: 4ab9bbcc-bdd4-41f3-9ab7-1e403bd8a834
  Object ID: recon_d98a8d


In [8]:
// Get the PLY file using artifacts API
const plyData = await getPlyArtifact(sessionId, objectId);

console.log("\n>> The PLY file is now available for:");
console.log("  - 3D visualization in external tools (Blender, MeshLab, etc.)");
console.log("  - Further processing with 3D libraries");
console.log("  - Integration into 3D applications");

// Note: To visualize in the notebook, you would need a 3D visualization library
// For now, the file is retrieved and ready for use in external tools


Retrieving PLY artifact → recon_d98a8d
Session ID: 4ab9bbcc-bdd4-41f3-9ab7-1e403bd8a834
✓ Retrieved 923 bytes

>> The PLY file is now available for:
  - 3D visualization in external tools (Blender, MeshLab, etc.)
  - Further processing with 3D libraries
  - Integration into 3D applications


In [9]:
const IMAGE_URL_FURN = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/furniture-colorful.jpg";

console.log("Input image:", IMAGE_URL_FURN);
console.log("\n>> Generating multi-object 3D reconstruction...");

const resultMulti = await chatCompletion(
    "Generate a 3D reconstruction of the two chairs in the image, by first detecting them, segmenting them and then reconstructing",
    [IMAGE_URL_FURN],
    undefined,
    Recon3DResponseSchema
);

console.log("\n>> RESPONSE");
console.log(resultMulti.data);
console.log("\n>> 3D Reconstruction PLY file:", resultMulti.data.recon_path);

// Extract session ID and object ID
const sessionIdMulti = resultMulti.response.session_id;
const messageContentMulti = resultMulti.response.choices[0]?.message?.content || "";

// Try to extract objectId from message content (format: recon_XXXXXX)
let reconMatchMulti = messageContentMulti.match(/recon_[a-f0-9]{6}/);

// If not found in message content, try to extract from the URL path
if (!reconMatchMulti && resultMulti.data.recon_path) {
    // Extract UUID from URL path and convert to objectId format
    // URL format: .../artifacts/{sessionId}/{uuid}/filename.ply
    const urlMatch = resultMulti.data.recon_path.match(/\/artifacts\/[^\/]+\/([^\/]+)\//);
    if (urlMatch) {
        // Convert UUID to objectId format (take first 6 hex chars)
        const uuid = urlMatch[1].replace(/-/g, '');
        const hexId = uuid.substring(0, 6);
        reconMatchMulti = [`recon_${hexId}`];
    }
}

if (!sessionIdMulti) {
    throw new Error("Session ID not found in response");
}

if (!reconMatchMulti) {
    throw new Error("Object ID (recon_XXXXXX) not found in message content or URL");
}

const objectIdMulti = reconMatchMulti[0];
console.log("\n>> Extracted IDs:");
console.log(`  Session ID: ${sessionIdMulti}`);
console.log(`  Object ID: ${objectIdMulti}`);


Input image: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/furniture-colorful.jpg

>> Generating multi-object 3D reconstruction...

>> RESPONSE
{ recon_path: "recon_cf3eaa" }

>> 3D Reconstruction PLY file: recon_cf3eaa

>> Extracted IDs:
  Session ID: d7dc2ebc-3300-4aa4-8795-856771e3d5bc
  Object ID: recon_cf3eaa


In [10]:
// Get the PLY file using artifacts API
const plyDataMulti = await getPlyArtifact(sessionIdMulti, objectIdMulti);

console.log("\n>> Multi-object 3D reconstruction completed!");
console.log("The PLY file contains the reconstructed chairs and is ready for visualization.");


Retrieving PLY artifact → recon_cf3eaa
Session ID: d7dc2ebc-3300-4aa4-8795-856771e3d5bc
✓ Retrieved 923 bytes

>> Multi-object 3D reconstruction completed!
The PLY file contains the reconstructed chairs and is ready for visualization.


### 3. 3D Reconstruction from Video

For better results, provide a video of the scene. The model will automatically extract frames from different viewpoints to create a more robust 3D reconstruction.


In [11]:
const VIDEO_URL = "https://storage.googleapis.com/vlm-data-public-prod/web/videos/tunnel.mp4";

console.log("Input video:", VIDEO_URL);
console.log("\n>> Generating 3D reconstruction from video...");

const resultScene = await chatCompletion(
    "Generate a 3D reconstruction of the scene by sampling some frames from the video",
    undefined,
    VIDEO_URL,
    Recon3DResponseSchema
);

console.log("\n>> RESPONSE");
console.log(resultScene.data);
console.log("\n>> 3D Reconstruction PLY file:", resultScene.data.recon_path);

// Extract session ID and object ID
const sessionIdScene = resultScene.response.session_id;
const messageContentScene = resultScene.response.choices[0]?.message?.content || "";

// Try to extract objectId from message content (format: recon_XXXXXX)
let reconMatchScene = messageContentScene.match(/recon_[a-f0-9]{6}/);

// If not found in message content, try to extract from the URL path
if (!reconMatchScene && resultScene.data.recon_path) {
    // Extract UUID from URL path and convert to objectId format
    // URL format: .../artifacts/{sessionId}/{uuid}/filename.ply
    const urlMatch = resultScene.data.recon_path.match(/\/artifacts\/[^\/]+\/([^\/]+)\//);
    if (urlMatch) {
        // Convert UUID to objectId format (take first 6 hex chars)
        const uuid = urlMatch[1].replace(/-/g, '');
        const hexId = uuid.substring(0, 6);
        reconMatchScene = [`recon_${hexId}`];
    }
}

if (!sessionIdScene) {
    throw new Error("Session ID not found in response");
}

if (!reconMatchScene) {
    throw new Error("Object ID (recon_XXXXXX) not found in message content or URL");
}

const objectIdScene = reconMatchScene[0];
console.log("\n>> Extracted IDs:");
console.log(`  Session ID: ${sessionIdScene}`);
console.log(`  Object ID: ${objectIdScene}`);


Input video: https://storage.googleapis.com/vlm-data-public-prod/web/videos/tunnel.mp4

>> Generating 3D reconstruction from video...

>> RESPONSE
{ recon_path: "recon_539ef6" }

>> 3D Reconstruction PLY file: recon_539ef6

>> Extracted IDs:
  Session ID: cc144a13-4274-42b0-9a70-889088e8c5a9
  Object ID: recon_539ef6


In [12]:
// Get the PLY file using artifacts API
const plyDataScene = await getPlyArtifact(sessionIdScene, objectIdScene);

console.log("\n>> 3D reconstruction from video completed!");
console.log("The PLY file contains the reconstructed scene and is ready for visualization.");


Retrieving PLY artifact → recon_539ef6
Session ID: cc144a13-4274-42b0-9a70-889088e8c5a9
✓ Retrieved 933 bytes

>> 3D reconstruction from video completed!
The PLY file contains the reconstructed scene and is ready for visualization.


---

## Conclusion

This cookbook demonstrated the comprehensive 3D reconstruction capabilities of the **VLM Run Orion Agent API** using Node.js/TypeScript.

### Key Takeaways

1. **OpenAI-Compatible Interface**: The API follows the OpenAI chat completions format, making it easy to integrate with existing workflows and tools.
2. **Structured Outputs**: Use Zod schemas with `response_format` parameter to get type-safe, validated responses with automatic parsing.
3. **3D Reconstruction from Single Images**: Generate 3D models by inferring depth and geometry from a single input image.
4. **Multi-Object 3D Reconstruction**: Reconstruct multiple objects within a single image by first detecting and segmenting them.
5. **3D Reconstruction from Video**: Utilize videos to automatically extract frames and reconstruct a scene for more robust 3D models.
6. **Type Safety**: TypeScript and Zod provide compile-time and runtime type checking for better developer experience.

### Next Steps

- Explore the [VLM Run Documentation](https://docs.vlm.run) for more details
- Check out the [Agent API docs](https://docs.vlm.run/agents/introduction) for advanced features
- Join our [Discord community](https://discord.gg/AMApC2UzVY) for support
- Check out more examples in the [VLM Run Cookbook](https://docs.vlm.run/blog)
- Review the [VLM Run Node.js SDK](https://github.com/vlm-run/vlmrun-node-sdk) documentation

Happy building!
